# Берем готовый датасет lstm и с помощью bigquery собираем датасет для gnn

In [ ]:
# Установить клиентскую библиотеку
%pip install google-cloud-bigquery pandas


Note: you may need to restart the kernel to use updated packages.
zsh:1: command not found: gcloud


In [11]:
# Шаг 1: Авторизация (если не делал)
!gcloud auth application-default login --quiet

# Шаг 2: Установка проекта для квоты
!gcloud auth application-default set-quota-project ethereal-fraud

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=pX7FcscOStwU9Oag5FwTp5bC5kNI5a&access_type=offline&code_challenge=lg9aCtFynOgIMQwuEsaXOKWYf7Z5Ve0U_rmBrNq3Znk&code_challenge_method=S256


Credentials saved to file: [/Users/a1234/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Cannot find a quota project to add to ADC. You might receive a "quota exceeded" or "API not enabled" error. Run $ gcloud auth application-default set-quota-project to add a quota project.

Credentials saved to file: [/Users/a1234/.config/gcl

In [6]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../ethereal-fraud-1c4b93baea54.json"
os.environ["GOOGLE_CLOUD_PROJECT"] = "ethereal-fraud"  # замените на свой ID

# Загружает LSTM датасет


In [7]:
import pandas as pd
from google.cloud import bigquery

# 1. Считываем LSTM-датасет
lstm_df = pd.read_csv('./data/transaction_dataset.csv', usecols=['Address','FLAG'])
lstm_df.columns = ['id','label']
seed_addrs = set(lstm_df['id'])

In [14]:
import pandas as pd

# Читаем LSTM-адреса
lstm = pd.read_csv('./data/transaction_dataset.csv', usecols=['Address'])
addrs = lstm['Address'].unique().tolist()

# 2. Формируем строки для UNNEST
addr_lines = [f"    '{addr}'" for addr in addrs]
addr_block = ",\n".join(addr_lines)
# 3. Шаблон SQL-запроса с JOIN вместо IN-подзапросов
sql = f"""#standardSQL
WITH
  seed AS (
    SELECT addr AS node
    FROM UNNEST([
{addr_block}
    ]) AS addr
  ),
  level1_out AS (
    SELECT t.to_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN seed ON t.from_address = seed.node
  ),
  level1_in AS (
    SELECT t.from_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN seed ON t.to_address = seed.node
  ),
  level1 AS (
    SELECT node FROM level1_out
    UNION DISTINCT
    SELECT node FROM level1_in
  ),
  level2_out AS (
    SELECT t.to_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN level1 ON t.from_address = level1.node
  ),
  level2_in AS (
    SELECT t.from_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN level1 ON t.to_address = level1.node
  ),
  level2 AS (
    SELECT node FROM level2_out
    UNION DISTINCT
    SELECT node FROM level2_in
  ),
  nodes AS (
    SELECT node FROM seed
    UNION DISTINCT
    SELECT node FROM level1
    UNION DISTINCT
    SELECT node FROM level2
  ),
  edges AS (
    SELECT
      t.from_address AS src,
      t.to_address   AS dst,
      t.value        AS amount,
      UNIX_SECONDS(t.block_timestamp) AS timestamp
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN nodes ON t.from_address = nodes.node
    JOIN nodes AS n2 ON t.to_address = n2.node
  )
SELECT * FROM edges;
"""


# Записываем в файл
with open('full_gnn_query.sql', 'w') as f:
    f.write(sql)

print("Полный SQL-запрос записан в full_gnn_query.sql")

Полный SQL-запрос записан в full_gnn_query.sql


In [ ]:
# -- вставьте сюда кусок из шага 2
# WITH seed AS (
#   SELECT addr AS node
#   FROM UNNEST([
#     '0xAAA…',
#     '0xBBB…',
#     -- …
#   ]) AS addr
# ),

# -- 0–2 уровень BFS
# RECURSIVE bfs AS (
#   SELECT node, 0 AS depth FROM seed

#   UNION ALL

#   SELECT t.to_address AS node, bfs.depth + 1
#     FROM `bigquery-public-data.crypto_ethereum.transactions` t
#     JOIN bfs ON t.from_address = bfs.node
#    WHERE bfs.depth < 2

#   UNION ALL

#   SELECT t.from_address AS node, bfs.depth + 1
#     FROM `bigquery-public-data.crypto_ethereum.transactions` t
#     JOIN bfs ON t.to_address = bfs.node
#    WHERE bfs.depth < 2
# ),

# nodes AS (
#   SELECT DISTINCT node FROM bfs
# ),

# -- все рёбра внутри подграфа
# edges AS (
#   SELECT
#     t.from_address AS src,
#     t.to_address   AS dst,
#     t.value        AS amount,
#     UNIX_SECONDS(t.block_timestamp) AS timestamp
#   FROM `bigquery-public-data.crypto_ethereum.transactions` t
#   JOIN nodes n1 ON t.from_address = n1.node
#   JOIN nodes n2 ON t.to_address   = n2.node
# )

# SELECT * FROM edges;

# Etherscan

In [ ]:
import time
import requests
import pandas as pd

API_KEY = ""
BASE_URL = "https://api.etherscan.io/api"

def fetch_txs(address, page=1, offset=10000):
    """Загружает транзакции (обычные) для address."""
    params = {
        "module": "account",
        "action": "txlist",
        "address": address,
        "startblock": 0,
        "endblock": 99999999,
        "page": page,
        "offset": offset,
        "sort": "desc",
        "apikey": API_KEY
    }
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    if data["status"] != "1":
        return []
    return data["result"]

def fetch_internal_txs(address, page=1, offset=10000):
    """Загружает внутренние (internal) транзакции."""
    params = {
        "module": "account",
        "action": "txlistinternal",
        "address": address,
        "startblock": 0,
        "endblock": 99999999,
        "page": page,
        "offset": offset,
        "sort": "desc",
        "apikey": API_KEY
    }
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    if data["status"] != "1":
        return []
    return data["result"]

def get_all_txs(address):
    """Собирает обычные + internal, пагинируя до пустой страницы."""
    all_txs = []
    for fetcher in (fetch_txs, fetch_internal_txs):
        page = 1
        while True:
            txs = fetcher(address, page=page)
            if not txs:
                break
            all_txs.extend(txs)
            page += 1
            time.sleep(0.2)   # чтобы не превысить 5 rps
    return all_txs

# 1) Читаем LSTM-адреса
lstm = pd.read_csv("./data/transaction_dataset.csv", usecols=["Address","FLAG"])
seed_addrs = set(lstm["Address"].tolist())

# 2) BFS-обход глубины 2
visited = set(seed_addrs)
frontier = set(seed_addrs)
edges = []   # тут будем собирать ребра

for depth in range(2):
    next_frontier = set()
    for addr in frontier:
        txs = get_all_txs(addr)
        for tx in txs:
            src = tx.get("from")
            dst = tx.get("to")
            amt = tx.get("value")
            ts  = int(tx.get("timeStamp", 0))
            edges.append((src, dst, amt, ts))
            # запоминаем нового соседа
            for nbr in (src, dst):
                if nbr not in visited:
                    visited.add(nbr)
                    next_frontier.add(nbr)
    frontier = next_frontier

# 3) Сохраняем transaction.csv
tx_df = pd.DataFrame(edges, columns=["src","dst","amount","timestamp"])
tx_df.to_csv("./data/transaction.csv", index=False)

# 4) Сбор account.csv
all_nodes = pd.Series(list(tx_df["src"]) + list(tx_df["dst"]), name="id")
all_nodes = all_nodes.drop_duplicates().to_frame()
# маппим метки: LSTM→FLAG, новые узлы = -1
label_map = dict(zip(lstm["Address"], lstm["FLAG"]))
all_nodes["label"] = all_nodes["id"].map(label_map).fillna(-1).astype(int)
all_nodes.to_csv("./data/account.csv", index=False)

print("Собрано:", tx_df.shape[0], "транзакций;", all_nodes.shape[0], "узлов.")